In [1]:
# OMDb API Key: bd1e1356

In [5]:
import requests
import pandas as pd
import pprint

url = "https://www.omdbapi.com/"
params = {
    "apikey": "bd1e1356",
    "t": "Avengers",
    "y": "2012",
    "type": "movie"
}

response = requests.get(url, params=params)
response.json()

{'Title': 'The Avengers',
 'Year': '2012',
 'Rated': 'PG-13',
 'Released': '04 May 2012',
 'Runtime': '143 min',
 'Genre': 'Action, Sci-Fi',
 'Director': 'Joss Whedon',
 'Writer': 'Joss Whedon, Zak Penn',
 'Actors': 'Robert Downey Jr., Chris Evans, Scarlett Johansson',
 'Plot': "Earth's mightiest heroes must come together and learn to fight as a team if they are going to stop the mischievous Loki and his alien army from enslaving humanity.",
 'Language': 'English, Russian',
 'Country': 'United States',
 'Awards': 'Nominated for 1 Oscar. 40 wins & 81 nominations total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BNGE0YTVjNzUtNzJjOS00NGNlLTgxMzctZTY4YTE1Y2Y1ZTU4XkEyXkFqcGc@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.0/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '91%'},
  {'Source': 'Metacritic', 'Value': '69/100'}],
 'Metascore': '69',
 'imdbRating': '8.0',
 'imdbVotes': '1,547,278',
 'imdbID': 'tt0848228',
 'Type': 'movie',
 'DVD': 'N/A',
 

In [7]:
import requests
import pandas as pd
import pprint

url = "https://www.omdbapi.com/"
params = {
    "apikey": "bd1e1356",
    "s": "Avengers",
    "type": "movie"
}

response = requests.get(url, params=params)
data = response.json()

In [10]:
movies = data["Search"]
for movie in movies:
  print("[", movie['Title'],"]",movie['Year'])

[ The Avengers ] 2012
[ Avengers: Endgame ] 2019
[ Avengers: Infinity War ] 2018
[ Avengers: Age of Ultron ] 2015
[ The Avengers ] 1998
[ Ultimate Avengers: The Movie ] 2006
[ Ultimate Avengers II ] 2006
[ Next Avengers: Heroes of Tomorrow ] 2008
[ Avengers Confidential: Black Widow & Punisher ] 2014
[ Crippled Avengers ] 1978


In [15]:
import pandas as pd
if 'Search' in data:
  m_df = pd.DataFrame(data['Search'])
  m_df
else:
  m_df = pd.DataFrame([data])
display(m_df)

,Title,Year,imdbID,Type,Poster
0,The Avengers,2012,tt0848228,movie,https://m.media-amazon.com/images/M/MV5BNGE0YT...
1,Avengers: Endgame,2019,tt4154796,movie,https://m.media-amazon.com/images/M/MV5BMTc5MD...
2,Avengers: Infinity War,2018,tt4154756,movie,https://m.media-amazon.com/images/M/MV5BMjMxNj...
3,Avengers: Age of Ultron,2015,tt2395427,movie,https://m.media-amazon.com/images/M/MV5BODBhYT...
4,The Avengers,1998,tt0118661,movie,https://m.media-amazon.com/images/M/MV5BZTA4Zm...
5,Ultimate Avengers: The Movie,2006,tt0491703,movie,https://m.media-amazon.com/images/M/MV5BMTYyMj...
6,Ultimate Avengers II,2006,tt0803093,movie,https://m.media-amazon.com/images/M/MV5BZjI3MT...
7,Next Avengers: Heroes of Tomorrow,2008,tt1259998,movie,https://m.media-amazon.com/images/M/MV5BMTQ3Nj...
8,Avengers Confidential: Black Widow & Punisher,2014,tt3482378,movie,https://m.media-amazon.com/images/M/MV5BMTU2MD...
9,Crippled Avengers,1978,tt0077292,movie,https://m.media-amazon.com/images/M/MV5BYjU2OD...


In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re # For cleaning text

In [18]:
# 1. 네이버 금융 종목 페이지 URL 만들기
def get_stock_info(code):
    url = f"https://finance.naver.com/item/main.naver?code={code}"

    # 2. requests.get()으로 HTML 가져오기 (인코딩: euc-kr)
    response = requests.get(url)
    response.encoding = 'euc-kr'
    html = response.text

    # 3. 뷰티플수프로 HTML 파싱
    soup = BeautifulSoup(html, 'html.parser')

    # 4. find()로 현재가 태그 찾아 텍스트 추출
    current_price_tag = soup.find('p', class_='no_today')
    current_price_text = current_price_tag.find('span', class_='blind').get_text(strip=True) if current_price_tag else 'N/A'
    current_price = int(current_price_text.replace(',', '')) if current_price_text != 'N/A' else None

    # 5. find()로 시세 테이블 태그 찾기
    no_info_table = soup.find('table', class_='no_info')
    stock_details = {}

    if no_info_table:
        # 6. find_all()로 <td> 태그 목록 가져오기
        td_tags = no_info_table.find_all('td')

        # Define the order of keys for the dictionary
        keys_order = ['전일', '시가', '고가', '저가', '거래량']

        # Iterate and extract specific details
        current_key_idx = 0
        for td in td_tags:
            em_tag = td.find('em')
            if em_tag:
                # Get the text, split by whitespace, take the first part, and remove commas
                raw_text = em_tag.get_text(strip=True)
                parts = raw_text.split()
                value_text = parts[0].replace(',', '')

                try:
                    value = int(value_text)
                except ValueError:
                    value = None

                # Assign values to predefined keys based on their order in the table
                if current_key_idx < len(keys_order):
                    stock_details[keys_order[current_key_idx]] = value
                    current_key_idx += 1


    # 8. 딕셔너리로 정리 후 반환
    result = {
        '현재가': current_price,
    }
    result.update(stock_details)
    return result

In [19]:
# 삼성전자 (종목코드: 005930) 정보 가져오기
samsung_info = get_stock_info('005930')
print("삼성전자 주가 정보:")
for key, value in samsung_info.items():
    print(f"{key}: {value:,}")

삼성전자 주가 정보:
현재가: 173,500
전일: 188,200,188,200
시가: 175,500,175,500
고가: 4,306,602,043,066,020
저가: 173,500,173,500
거래량: 167,300,167,300


In [20]:
# 여러 종목 주가 한번에 수집: apply() 메서드

# 예시 종목 리스트
stocks = pd.DataFrame({
    'name': ['삼성전자', 'SK하이닉스', 'NAVER', 'LG에너지솔루션'],
    'code': ['005930', '000660', '035420', '373220']
})

print("\n--- 여러 종목 주가 정보 수집 ---")
# `apply()` 메서드를 사용하여 각 종목의 정보를 가져옵니다.
# `result_series`는 각 행에 대해 `get_stock_info` 함수의 결과(딕셔너리)를 담는 Series가 됩니다.
stock_details_series = stocks.apply(lambda row: get_stock_info(row['code']), axis=1)

# Series of dictionaries를 DataFrame으로 변환
stock_details_df = pd.DataFrame(stock_details_series.tolist())

# 기존 stocks DataFrame과 합치기 (merge() 사용)
# 여기서는 단순히 stocks DataFrame에 새로운 열을 추가하는 방식으로도 가능하지만,
# merge() 사용법을 보여주기 위해 별도의 DataFrame을 생성 후 합칩니다.
final_df = pd.merge(stocks, stock_details_df, left_index=True, right_index=True)

display(final_df)


--- 여러 종목 주가 정보 수집 ---


,name,code,현재가,전일,시가,고가,저가,거래량
0,삼성전자,005930,173500,188200188200,175500175500,4306602043066020,173500173500,167300167300
1,SK하이닉스,000660,836000,924000924000,864000864000,72535517253551,851000851000,808000808000
2,NAVER,035420,218500,222500222500,218500218500,940462940462,209500209500,207000207000
3,LG에너지솔루션,373220,359500,377500377500,364500364500,414051414051,360000360000,350500350500
